In [9]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import webbrowser
from statsmodels.tsa.api import VAR

# ============================================================
# 0. Renderer / browser block
# ============================================================
# VS Code 내부 렌더러 사용
pio.renderers.default = "vscode"

# 브라우저 강제 차단
webbrowser.open = lambda *args, **kwargs: None

# ============================================================
# 1. Config
# ============================================================
BASE_DIR = Path("./")
RESULT_DIR = BASE_DIR / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

EVENT_FILE = BASE_DIR / "event_clusters_final.csv"
DATA_FILE = BASE_DIR / "tvpvar_raw_level_merged.csv"

ROLLING_WINDOW = 60
LAG = 2
H = 10
DC_Q = 0.75

# ============================================================
# 2. Event label mapping
# ============================================================
event_label_map = {
    "2020-11-09": "코로나 백신 발표로 인한 경기 회복 기대, 금/은 안전자산 가격 급락",
    "2024-08-05": "미국 경기둔화·침체 우려가 촉발한 글로벌 위험자산 매도",
    "2024-12-18": "Fed 정책결정 대기와 금리경로 재평가",
    "2025-07-08": "트럼프의 구리 50% 관세 방침 발표",
    "2025-07-31": "미국 정제동을 관세 대상에서 제외",
    "2020-10-22": "미국 경기지표 개선과 달러 반등으로 금·은 약세",
    "2020-10-28": "유럽·미국의 코로나 재확산과 봉쇄 우려",
    "2020-11-04": "미국 대선 개표",
    "2021-02-01": "실버 스퀴즈 발생",
    "2021-06-17": "FOMC 이후 달러 급등과 조기 긴축 기대 강화",
    "2022-03-01": "러시아-우크라이나 전쟁 초기 충격",
    "2022-03-07": "전쟁 충격이 더 커지며 LME 금속시장 전반 영향",
    "2022-06-13": "미국 CPI 충격 직후 인플레이션과 긴축 우려",
    "2022-07-28": "Recession fear",
    "2022-11-10": "미국 CPI가 예상보다 낮게 나오며 Fed 속도조절 기대",
    "2022-11-17": "달러 반등과 Fed의 추가 긴축 발언으로 금이 후퇴",
    "2022-12-20": "중국 리오프닝 기대",
    "2023-01-03": "중국 리오프닝과 달러 약세 기대",
    "2023-05-11": "중국 수요 둔화·미국 부채한도 불안, 생성형 AI 인프라 투자 확대 기대",
    "2023-10-02": "파나마의 First Quantum 신규 광산계약 논의 중단",
    "2023-10-16": "Geopolitical safe-haven bid",
    "2024-02-15": "미국 인플레이션·소매판매 지표로 인해 rate cut 기대 흔들림",
    "2024-04-10": "중국 경기지표 개선과 원료 부족 우려로 강세",
    "2024-04-30": "AI·데이터센터가 2030년까지 구리 수요를 100만 톤 추가할 수 있다는 전망",
    "2024-11-06": "미국 대선에서 트럼프 승리가 확정",
    "2025-01-13": "LME가 2022 니켈 위기 후 거래 회복을 확인",
    "2025-02-03": "무역불안에 따른 금 강세, 구조적 구리 수요 테마 명확",
    "2025-04-03": "트럼프의 공격적 관세 발표로 금은 사상 최고치, 구리는 경기둔화 우려로 약세",
    "2025-11-13": "미국 셧다운 종료와 더 비둘기파적인 금리 기대 속 금과 은이 상승",
    "2025-12-29": "금·은이 급등 후 변동성 확대, 구리는 글로벌 공급부족 우려로 15년 만의 최대 연간 상승폭",
    "2022-03-02": "러-우 전쟁 초기, 안전자산 확대",
    "2022-09-13": "미국 CPI 쇼크 (인플레이션 재상승)",
    "2022-12-15": "Fed 금리 경로 경계 속에서도 중국 리오프닝 기대",
    "2025-04-08": "미·중 무역갈등 재격화와 관세 불안, AI와 DC가 2030년까지 구리 수요를 100만 톤 추가",
    "2025-07-09": "트럼프의 구리 50% 관세 방침 발표",
    "2025-09-02": "금은 달러·장기금리·재정 불안 속에서 사상 최고권, 구리가 달러 약세와 중국 지표를 바탕으로 2개월 고점",
    "2025-11-17": "중앙은행의 지속적 금 매수 전망",
    "2025-11-24": "UBS의 구리 전망 상향",
}

# cluster color
cluster_color_map = {
    "Structural Persistent": "#1f77b4",
    "Transient Shock": "#d62728",
    "Delayed Response": "#ff7f0e",
    "Minor / Local": "#7f7f7f",
}

# ============================================================
# 3. Load
# ============================================================
print("[1/7] Load")

df_event = pd.read_csv(EVENT_FILE)
df_event["Date"] = pd.to_datetime(df_event["Date"])
df_event = df_event.sort_values("Date").reset_index(drop=True)
df_event["DateStr"] = df_event["Date"].dt.strftime("%Y-%m-%d")
df_event["MappedEventLabel"] = df_event["DateStr"].map(event_label_map).fillna("No mapped label")

df = pd.read_csv(DATA_FILE)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# ============================================================
# 4. Returns
# ============================================================
print("[2/7] Returns")

for col in ["SOLVPN", "COPPER", "GOLD", "SILVER", "DXY", "VIX"]:
    df[f"dlog_{col}"] = np.log(df[col]).diff()

df["d_UST10Y"] = df["UST10Y"].diff()

VAR_COLS = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "d_UST10Y",
    "dlog_VIX",
]

df = df.dropna(subset=VAR_COLS).reset_index(drop=True)

# ============================================================
# 5. Rolling FEVD
# ============================================================
print("[3/7] FEVD")

def compute_fevd(data: pd.DataFrame) -> np.ndarray:
    model = VAR(data)
    res = model.fit(LAG)
    return res.fevd(H).decomp[-1]

k = len(VAR_COLS)
fevd_list = []
dates = []

for t in range(ROLLING_WINDOW, len(df)):
    window = df.iloc[t - ROLLING_WINDOW:t][VAR_COLS]
    try:
        fevd = compute_fevd(window)
    except Exception:
        fevd = np.full((k, k), np.nan)

    fevd_list.append(fevd)
    dates.append(df.loc[t, "Date"])

fevd_array = np.array(fevd_list)
dates = pd.Series(dates)

i_dc = VAR_COLS.index("dlog_SOLVPN")
i_cp = VAR_COLS.index("dlog_COPPER")

# ============================================================
# 6. DC relevance score
# ============================================================
print("[4/7] Scoring")

rows = []

for _, row in df_event.iterrows():
    d = row["Date"]
    idx = int(np.argmin(np.abs(dates - d)))
    fevd = fevd_array[idx]

    if np.isnan(fevd).all():
        continue

    off = fevd.copy()
    np.fill_diagonal(off, 0.0)
    total = np.nansum(off)

    dc_strength = sum(
        abs(fevd[j, i_dc]) + abs(fevd[i_dc, j])
        for j in range(k) if j != i_dc
    )
    dc_cp = abs(fevd[i_cp, i_dc]) + abs(fevd[i_dc, i_cp])

    rows.append({
        "Date": d,
        "DateStr": d.strftime("%Y-%m-%d"),
        "MappedEventLabel": row["MappedEventLabel"],
        "Cluster": row["Cluster"],
        "ClusterLabel": row["ClusterLabel"],
        "DC_relevance": dc_strength / total if total != 0 else np.nan,
        "DC_COPPER_relevance": dc_cp / total if total != 0 else np.nan,
        "SOLVPN_to_COPPER": fevd[i_cp, i_dc],
        "COPPER_to_SOLVPN": fevd[i_dc, i_cp],
    })

score = pd.DataFrame(rows).sort_values("Date").reset_index(drop=True)

threshold = score["DC_relevance"].quantile(DC_Q)
dc_events = score[score["DC_relevance"] >= threshold].copy().reset_index(drop=True)
dc_events["DC_Event_Type"] = "DC-focused"

score.to_csv(RESULT_DIR / "dc_event_scores.csv", index=False, encoding="utf-8-sig")
dc_events.to_csv(RESULT_DIR / "dc_filtered_events.csv", index=False, encoding="utf-8-sig")

print(f"[INFO] Total scored events: {len(score)}")
print(f"[INFO] DC filtered events: {len(dc_events)}")
print(f"[INFO] Threshold (q={DC_Q:.2f}): {threshold:.4f}")

# ============================================================
# 7. Spillover time series
# ============================================================
print("[5/7] Build time series")

dc_to_cp = []
cp_to_dc = []
pair_total = []
asymmetry = []

for fevd in fevd_array:
    if np.isnan(fevd).all():
        dc_to_cp.append(np.nan)
        cp_to_dc.append(np.nan)
        pair_total.append(np.nan)
        asymmetry.append(np.nan)
        continue

    v_dc_cp = fevd[i_cp, i_dc]
    v_cp_dc = fevd[i_dc, i_cp]

    dc_to_cp.append(v_dc_cp)
    cp_to_dc.append(v_cp_dc)
    pair_total.append(abs(v_dc_cp) + abs(v_cp_dc))
    asymmetry.append(v_dc_cp - v_cp_dc)

spill_df = pd.DataFrame({
    "Date": dates,
    "SOLVPN_to_COPPER": dc_to_cp,
    "COPPER_to_SOLVPN": cp_to_dc,
    "PairTotal": pair_total,
    "Asymmetry": asymmetry,
}).sort_values("Date").reset_index(drop=True)

ret_df = df[["Date", "dlog_SOLVPN", "dlog_COPPER"]].copy()

# ============================================================
# 8. Interactive figure
# ============================================================
print("[6/7] Plot")

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=[
        "SOLVPN ↔ COPPER Directional Spillover",
        "Pair Total / Asymmetry",
        "SOLVPN and COPPER Returns",
        "Event Type Timeline",
    ],
)

# Row 1
fig.add_trace(
    go.Scatter(
        x=spill_df["Date"],
        y=spill_df["SOLVPN_to_COPPER"],
        mode="lines",
        name="SOLVPN → COPPER",
        line=dict(width=2),
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=spill_df["Date"],
        y=spill_df["COPPER_to_SOLVPN"],
        mode="lines",
        name="COPPER → SOLVPN",
        line=dict(width=2),
    ),
    row=1, col=1
)

# Row 2
fig.add_trace(
    go.Scatter(
        x=spill_df["Date"],
        y=spill_df["PairTotal"],
        mode="lines",
        name="PairTotal",
        line=dict(width=2),
    ),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=spill_df["Date"],
        y=spill_df["Asymmetry"],
        mode="lines",
        name="Asymmetry",
        line=dict(width=2, dash="dash"),
    ),
    row=2, col=1
)

# Row 3
fig.add_trace(
    go.Scatter(
        x=ret_df["Date"],
        y=ret_df["dlog_SOLVPN"],
        mode="lines",
        name="dlog_SOLVPN",
        line=dict(width=1.8),
    ),
    row=3, col=1
)
fig.add_trace(
    go.Scatter(
        x=ret_df["Date"],
        y=ret_df["dlog_COPPER"],
        mode="lines",
        name="dlog_COPPER",
        line=dict(width=1.8),
    ),
    row=3, col=1
)

# Row 4: event type timeline
y_map = {
    "Structural Persistent": 4,
    "Transient Shock": 3,
    "Delayed Response": 2,
    "Minor / Local": 1,
}

for label, y in y_map.items():
    sub = score[score["ClusterLabel"] == label].copy()
    if len(sub) == 0:
        continue

    fig.add_trace(
        go.Scatter(
            x=sub["Date"],
            y=[y] * len(sub),
            mode="markers",
            name=f"Event Type: {label}",
            marker=dict(
                size=10,
                color=cluster_color_map.get(label, "#444444"),
                symbol="circle",
                line=dict(width=0.5, color="white"),
            ),
            customdata=np.stack([
                sub["DateStr"].astype(str),
                sub["MappedEventLabel"].astype(str),
                sub["ClusterLabel"].astype(str),
                sub["DC_relevance"].round(3).astype(str),
                sub["DC_COPPER_relevance"].round(3).astype(str),
            ], axis=1),
            hovertemplate=(
                "Date=%{customdata[0]}<br>"
                "Event=%{customdata[1]}<br>"
                "Type=%{customdata[2]}<br>"
                "DC_relevance=%{customdata[3]}<br>"
                "DC_COPPER_relevance=%{customdata[4]}"
                "<extra></extra>"
            ),
        ),
        row=4, col=1
    )

# vertical lines for filtered DC events
for _, r in dc_events.iterrows():
    d = r["Date"]
    fig.add_vline(x=d, line_dash="dot", line_color="red", opacity=0.45)

    hover_txt = (
        f"Date={r['DateStr']}<br>"
        f"Event={r['MappedEventLabel']}<br>"
        f"Cluster=C{int(r['Cluster'])} | {r['ClusterLabel']}<br>"
        f"Type=DC-focused<br>"
        f"DC_relevance={r['DC_relevance']:.3f}<br>"
        f"DC_COPPER_relevance={r['DC_COPPER_relevance']:.3f}<br>"
        f"SOLVPN→COPPER={r['SOLVPN_to_COPPER']:.3f}<br>"
        f"COPPER→SOLVPN={r['COPPER_to_SOLVPN']:.3f}"
    )

    y1 = spill_df["SOLVPN_to_COPPER"].max()
    y2 = spill_df["PairTotal"].max()
    y3 = ret_df["dlog_SOLVPN"].max()
    y4 = 4.35

    for rr, yy in [(1, y1), (2, y2), (3, y3), (4, y4)]:
        fig.add_trace(
            go.Scatter(
                x=[d],
                y=[yy],
                mode="markers",
                marker=dict(size=9, color="red", symbol="diamond"),
                showlegend=False,
                hovertemplate=hover_txt + "<extra></extra>",
            ),
            row=rr, col=1
        )

fig.update_yaxes(title_text="Directional Spillover", row=1, col=1)
fig.update_yaxes(title_text="Pair Structure", row=2, col=1)
fig.update_yaxes(title_text="Return", row=3, col=1)
fig.update_yaxes(
    title_text="Event Type",
    row=4, col=1,
    tickmode="array",
    tickvals=[1, 2, 3, 4],
    ticktext=["Minor / Local", "Delayed Response", "Transient Shock", "Structural Persistent"],
    range=[0.5, 4.7],
)
fig.update_xaxes(title_text="Date", row=4, col=1)

fig.update_layout(
    height=1100,
    title="Data Center-focused Events from Existing Event Set: SOLVPN ↔ COPPER Spillover Structure",
    hovermode="x unified",
)

print("[7/7] Show")
fig.show()

print("\n[DONE] Saved files:")
print("-", RESULT_DIR / "dc_event_scores.csv")
print("-", RESULT_DIR / "dc_filtered_events.csv")

[1/7] Load
[2/7] Returns
[3/7] FEVD
[4/7] Scoring
[INFO] Total scored events: 38
[INFO] DC filtered events: 10
[INFO] Threshold (q=0.75): 0.3406
[5/7] Build time series
[6/7] Plot
[7/7] Show



[DONE] Saved files:
- result\dc_event_scores.csv
- result\dc_filtered_events.csv
